# Advanced Python Properties — Problems with Complete Solutions

This notebook focuses on advanced uses of Python properties:

- read-only properties
- computed properties
- validation with setters
- cache invalidation
- dependent computed properties
- lazy computation
- lazy resource loading
- avoiding stale cached data
- `functools.cached_property`
- explicit invalidation APIs
- inheritance and abstract properties
- property introspection
- thread-safe lazy caching
- design trade-offs between properties and methods

The problems are intentionally more difficult than basic `@property` exercises. Each problem includes a complete reference solution, checks/tests, and optional extensions.

> **Best-practice theme:** a property should usually *feel like attribute access*. If obtaining a value is slow, performs network I/O, mutates external state, or can fail for many environmental reasons, a method may communicate the cost more clearly.

## 0. Warm-up Example — A Read-Only Computed Property

A circle's area does not need to be stored separately from its radius. It can be computed from the radius whenever needed.

The key idea is that a getter-only property is read-only through the normal public interface.

In [1]:
from math import pi

class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):
        return pi * self.radius ** 2


c = Circle(2)
print(c.area)
assert c.area == pi * 4

# Uncommenting this should raise AttributeError:
# c.area = 100

12.566370614359172


# Problem 1 — Validated Radius + Read-Only Geometry

Implement a `Circle` class with these requirements:

1. `radius` is readable and writable.
2. `radius` must be a real number but not a boolean.
3. `radius` must be non-negative.
4. `diameter`, `circumference`, and `area` are read-only computed properties.
5. Changing `radius` must immediately affect all computed values.

### Why this is useful

The backing state is only the radius. Everything else is derived data, so storing all four values would create synchronization problems.

In [2]:
from math import pi
from numbers import Real

class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if isinstance(value, bool) or not isinstance(value, Real):
            raise TypeError("radius must be a real number")
        if value < 0:
            raise ValueError("radius must be non-negative")
        self._radius = value

    @property
    def diameter(self):
        return 2 * self.radius

    @property
    def circumference(self):
        return 2 * pi * self.radius

    @property
    def area(self):
        return pi * self.radius ** 2


c = Circle(3)

assert c.radius == 3
assert c.diameter == 6
assert c.circumference == 6 * pi
assert c.area == 9 * pi

c.radius = 5

assert c.diameter == 10
assert c.area == 25 * pi

for bad in (-1, -10.5):
    try:
        c.radius = bad
    except ValueError:
        pass
    else:
        raise AssertionError("negative radius should fail")

for bad in ("3", None, True):
    try:
        c.radius = bad
    except TypeError:
        pass
    else:
        raise AssertionError("non-real radius should fail")

print("Problem 1 passed")

Problem 1 passed


### Extension ideas

- Add a writable `diameter` property that updates `radius`.
- Decide whether `Decimal` or `Fraction` should be accepted.
- Decide how NaN and infinity should be handled.

# Problem 2 — Cached Computed Property with Correct Invalidation

Computing a value can be expensive. Build a `Circle` whose `area` is cached after the first access.

Requirements:

1. The first `area` access computes the area.
2. Repeated access without changing `radius` reuses the cache.
3. Assigning the *same radius* should not invalidate the cache.
4. Assigning a different radius invalidates the cache.
5. Expose a read-only `area_computation_count` property for testing.

In [3]:
from math import pi
from numbers import Real

class CachedCircle:
    def __init__(self, radius):
        self._radius = None
        self._area_cache = None
        self._area_computation_count = 0
        self.radius = radius

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if isinstance(value, bool) or not isinstance(value, Real):
            raise TypeError("radius must be a real number")
        if value < 0:
            raise ValueError("radius must be non-negative")

        if value != self._radius:
            self._radius = value
            self._area_cache = None

    @property
    def area(self):
        if self._area_cache is None:
            self._area_computation_count += 1
            self._area_cache = pi * self.radius ** 2
        return self._area_cache

    @property
    def area_computation_count(self):
        return self._area_computation_count


c = CachedCircle(2)

assert c.area_computation_count == 0
first = c.area
assert c.area_computation_count == 1

second = c.area
assert first == second
assert c.area_computation_count == 1

c.radius = 2
assert c.area == first
assert c.area_computation_count == 1

c.radius = 4
assert c.area == 16 * pi
assert c.area_computation_count == 2

print("Problem 2 passed")

Problem 2 passed


# Problem 3 — Multiple Dependent Caches

Create a `Rectangle` with writable `width` and `height`.

Computed read-only properties:

- `area`
- `perimeter`
- `diagonal`

All three values are cached independently.

Requirements:

1. Changing either dimension invalidates all geometry caches.
2. Setting a dimension to its current value does not invalidate caches.
3. Keep the invalidation logic in one helper method.
4. Validate dimensions as strictly positive real numbers.

In [4]:
from math import hypot
from numbers import Real

class Rectangle:
    def __init__(self, width, height):
        self._width = None
        self._height = None
        self._area_cache = None
        self._perimeter_cache = None
        self._diagonal_cache = None

        self.width = width
        self.height = height

    @staticmethod
    def _validate_dimension(name, value):
        if isinstance(value, bool) or not isinstance(value, Real):
            raise TypeError(f"{name} must be a real number")
        if value <= 0:
            raise ValueError(f"{name} must be > 0")

    def _invalidate_geometry_cache(self):
        self._area_cache = None
        self._perimeter_cache = None
        self._diagonal_cache = None

    @property
    def width(self):
        return self._width

    @width.setter
    def width(self, value):
        self._validate_dimension("width", value)
        if value != self._width:
            self._width = value
            self._invalidate_geometry_cache()

    @property
    def height(self):
        return self._height

    @height.setter
    def height(self, value):
        self._validate_dimension("height", value)
        if value != self._height:
            self._height = value
            self._invalidate_geometry_cache()

    @property
    def area(self):
        if self._area_cache is None:
            self._area_cache = self.width * self.height
        return self._area_cache

    @property
    def perimeter(self):
        if self._perimeter_cache is None:
            self._perimeter_cache = 2 * (self.width + self.height)
        return self._perimeter_cache

    @property
    def diagonal(self):
        if self._diagonal_cache is None:
            self._diagonal_cache = hypot(self.width, self.height)
        return self._diagonal_cache


r = Rectangle(3, 4)
assert r.area == 12
assert r.perimeter == 14
assert r.diagonal == 5

# Force all caches to be populated.
_ = (r.area, r.perimeter, r.diagonal)

old_cache_ids = (
    id(r._area_cache),
    id(r._perimeter_cache),
    id(r._diagonal_cache),
)

r.width = 6

assert r._area_cache is None
assert r._perimeter_cache is None
assert r._diagonal_cache is None

assert r.area == 24
assert r.perimeter == 20
assert r.diagonal == hypot(6, 4)

print("Problem 3 passed")

Problem 3 passed


# Problem 4 — Sentinel Objects: Caching `None` Correctly

A common cache bug appears when `None` is itself a valid computed result.

Suppose a property looks up a user by ID. If the user does not exist, the correct result is `None`. Using `None` to also mean “not computed yet” causes repeated lookups.

Implement `UserReference.user` so that:

1. The lookup runs at most once until invalidated.
2. A legitimate `None` result is cached.
3. Changing `user_id` invalidates the cached result.
4. The lookup dependency is injected as a callable, which makes the class testable.

In [5]:
_NOT_COMPUTED = object()

class UserReference:
    def __init__(self, user_id, lookup):
        if not callable(lookup):
            raise TypeError("lookup must be callable")
        self._lookup = lookup
        self._user_id = None
        self._user_cache = _NOT_COMPUTED
        self.user_id = user_id

    @property
    def user_id(self):
        return self._user_id

    @user_id.setter
    def user_id(self, value):
        if not isinstance(value, int) or isinstance(value, bool):
            raise TypeError("user_id must be an int")
        if value <= 0:
            raise ValueError("user_id must be positive")

        if value != self._user_id:
            self._user_id = value
            self._user_cache = _NOT_COMPUTED

    @property
    def user(self):
        if self._user_cache is _NOT_COMPUTED:
            self._user_cache = self._lookup(self.user_id)
        return self._user_cache


calls = []

def fake_lookup(user_id):
    calls.append(user_id)
    data = {
        1: {"id": 1, "name": "Ada"},
        2: None,
    }
    return data.get(user_id)

ref = UserReference(2, fake_lookup)

assert ref.user is None
assert ref.user is None
assert calls == [2]  # None was cached correctly.

ref.user_id = 1
assert ref.user == {"id": 1, "name": "Ada"}
assert calls == [2, 1]

print("Problem 4 passed")

Problem 4 passed


# Problem 5 — Avoiding Mutable Cache Leaks

A read-only property can still expose mutable internal state.

Consider a class that computes a list of normalized tags. If the property returns the cached list itself, callers can mutate it and corrupt the object's cache.

Implement `Article.normalized_tags` so that:

1. Tags are stripped and lowercased.
2. Duplicate normalized tags are removed while preserving order.
3. The result is cached.
4. Callers cannot mutate the cached result.
5. Replacing `tags` invalidates the cache.

In [6]:
class Article:
    def __init__(self, tags):
        self._tags = ()
        self._normalized_tags_cache = None
        self.tags = tags

    @property
    def tags(self):
        # Return an immutable representation.
        return self._tags

    @tags.setter
    def tags(self, values):
        if isinstance(values, (str, bytes)):
            raise TypeError("tags must be an iterable of strings, not one string")

        values = tuple(values)

        if not all(isinstance(x, str) for x in values):
            raise TypeError("every tag must be a string")

        if values != self._tags:
            self._tags = values
            self._normalized_tags_cache = None

    @property
    def normalized_tags(self):
        if self._normalized_tags_cache is None:
            seen = set()
            result = []

            for tag in self.tags:
                normalized = tag.strip().lower()
                if normalized and normalized not in seen:
                    seen.add(normalized)
                    result.append(normalized)

            # Cache an immutable tuple.
            self._normalized_tags_cache = tuple(result)

        return self._normalized_tags_cache


a = Article([" Python ", "OOP", "python", "  Properties "])

assert a.normalized_tags == ("python", "oop", "properties")
assert isinstance(a.normalized_tags, tuple)

try:
    a.normalized_tags.append("broken")
except AttributeError:
    pass
else:
    raise AssertionError("tuple should prevent cache mutation")

a.tags = ["Cache", "CACHE", "Properties"]
assert a.normalized_tags == ("cache", "properties")

print("Problem 5 passed")

Problem 5 passed


# Problem 6 — Writable Derived Property

Sometimes a derived property can be meaningfully writable.

Implement a `Temperature` class with:

- internal canonical storage in Celsius
- writable `celsius`
- writable `fahrenheit`
- writable `kelvin`

Requirements:

1. Assigning any representation updates the same canonical Celsius value.
2. Kelvin cannot be below absolute zero.
3. Celsius cannot be below absolute zero.
4. Fahrenheit assignment must also respect absolute zero after conversion.
5. Use properties rather than duplicate state.

In [7]:
from numbers import Real

class Temperature:
    ABSOLUTE_ZERO_C = -273.15

    def __init__(self, *, celsius=0.0):
        self.celsius = celsius

    @staticmethod
    def _validate_number(value):
        if isinstance(value, bool) or not isinstance(value, Real):
            raise TypeError("temperature must be a real number")

    @classmethod
    def _validate_celsius(cls, value):
        cls._validate_number(value)
        if value < cls.ABSOLUTE_ZERO_C:
            raise ValueError("temperature cannot be below absolute zero")

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        self._validate_celsius(value)
        self._celsius = float(value)

    @property
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        self._validate_number(value)
        celsius = (value - 32) * 5 / 9
        self._validate_celsius(celsius)
        self._celsius = float(celsius)

    @property
    def kelvin(self):
        return self.celsius + 273.15

    @kelvin.setter
    def kelvin(self, value):
        self._validate_number(value)
        celsius = value - 273.15
        self._validate_celsius(celsius)
        self._celsius = float(celsius)


t = Temperature(celsius=100)
assert round(t.fahrenheit, 8) == 212
assert round(t.kelvin, 8) == 373.15

t.fahrenheit = 32
assert round(t.celsius, 8) == 0

t.kelvin = 273.15
assert round(t.celsius, 8) == 0

try:
    t.kelvin = -0.01
except ValueError:
    pass
else:
    raise AssertionError("negative Kelvin must fail")

print("Problem 6 passed")

Problem 6 passed


# Problem 7 — `cached_property` and Its Invalidation Trade-Off

Python's `functools.cached_property` stores the computed value on the instance after first access.

Build a `Dataset` where:

- `values` is mutable only by replacing the full sequence through a setter.
- `mean` uses `cached_property`.
- replacing `values` invalidates `mean`.

Then inspect `__dict__` to understand where the cached value lives.

In [8]:
from functools import cached_property
from numbers import Real

class Dataset:
    def __init__(self, values):
        self._values = ()
        self.values = values

    @property
    def values(self):
        return self._values

    @values.setter
    def values(self, new_values):
        new_values = tuple(new_values)

        if not new_values:
            raise ValueError("values cannot be empty")

        if any(isinstance(x, bool) or not isinstance(x, Real) for x in new_values):
            raise TypeError("all values must be real numbers")

        self._values = new_values

        # cached_property writes its cached value under the attribute name.
        self.__dict__.pop("mean", None)

    @cached_property
    def mean(self):
        print("computing mean...")
        return sum(self.values) / len(self.values)


d = Dataset([10, 20, 30])

assert "mean" not in d.__dict__
assert d.mean == 20
assert "mean" in d.__dict__

# Second access does not recompute.
assert d.mean == 20

d.values = [100, 200]
assert "mean" not in d.__dict__
assert d.mean == 150

print("Problem 7 passed")

computing mean...
computing mean...
Problem 7 passed


### Design note

`cached_property` is convenient when:

- the value depends only on stable instance state, or
- you have a clear invalidation strategy.

It is less suitable when many input attributes can change independently and invalidation becomes difficult to reason about.

# Problem 8 — Dependency-Aware Versioned Cache

Manual invalidation becomes tedious when many properties depend on the same mutable state.

Implement a `Vector` that uses a **version number**:

- Mutating `x` or `y` increments `_version`.
- The `magnitude` cache stores both a value and the version at which it was computed.
- `magnitude` recomputes only if the versions differ.

This pattern generalizes to more complicated dependency graphs.

In [9]:
from math import hypot
from numbers import Real

class Vector:
    def __init__(self, x, y):
        self._version = 0
        self._x = None
        self._y = None
        self._magnitude_cache = None
        self._magnitude_cache_version = -1

        self.x = x
        self.y = y

    @staticmethod
    def _validate(value):
        if isinstance(value, bool) or not isinstance(value, Real):
            raise TypeError("coordinate must be a real number")

    def _set_coordinate(self, attr_name, value):
        self._validate(value)

        if getattr(self, attr_name) != value:
            setattr(self, attr_name, value)
            self._version += 1

    @property
    def x(self):
        return self._x

    @x.setter
    def x(self, value):
        self._set_coordinate("_x", value)

    @property
    def y(self):
        return self._y

    @y.setter
    def y(self, value):
        self._set_coordinate("_y", value)

    @property
    def magnitude(self):
        if self._magnitude_cache_version != self._version:
            self._magnitude_cache = hypot(self.x, self.y)
            self._magnitude_cache_version = self._version
        return self._magnitude_cache


v = Vector(3, 4)
assert v.magnitude == 5

cached_version = v._magnitude_cache_version
assert cached_version == v._version

v.x = 3
assert v._version == cached_version  # no real change
assert v.magnitude == 5

v.x = 6
assert v._version > cached_version
assert v.magnitude == hypot(6, 4)

print("Problem 8 passed")

Problem 8 passed


# Problem 9 — Lazy Resource Loading Without Hard-Coding Network Access

A class often needs to lazily obtain expensive data. For testability, do not hard-code the external operation inside the property.

Build a `WebDocument` with an injected `fetcher(url)` callable.

Requirements:

1. `content` lazily fetches bytes only once per URL.
2. `size` and `text` depend on `content`.
3. Changing `url` invalidates all URL-dependent state.
4. `load_count` tracks actual fetch operations.
5. `refresh()` explicitly reloads the current URL.
6. URL validation occurs before state mutation.

In [10]:
class WebDocument:
    def __init__(self, url, fetcher, *, encoding="utf-8"):
        if not callable(fetcher):
            raise TypeError("fetcher must be callable")

        self._fetcher = fetcher
        self._encoding = encoding
        self._url = None
        self._content = None
        self._load_count = 0
        self.url = url

    @staticmethod
    def _validate_url(value):
        if not isinstance(value, str):
            raise TypeError("url must be a string")
        if not value.startswith(("http://", "https://")):
            raise ValueError("url must start with http:// or https://")

    def _invalidate(self):
        self._content = None

    @property
    def url(self):
        return self._url

    @url.setter
    def url(self, value):
        self._validate_url(value)
        if value != self._url:
            self._url = value
            self._invalidate()

    @property
    def content(self):
        if self._content is None:
            data = self._fetcher(self.url)
            if not isinstance(data, bytes):
                raise TypeError("fetcher must return bytes")
            self._content = data
            self._load_count += 1
        return self._content

    @property
    def size(self):
        return len(self.content)

    @property
    def text(self):
        return self.content.decode(self._encoding)

    @property
    def load_count(self):
        return self._load_count

    def refresh(self):
        self._invalidate()
        return self.content


responses = {
    "https://example.test/a": b"alpha",
    "https://example.test/b": b"beta-beta",
}

fetch_calls = []

def fake_fetcher(url):
    fetch_calls.append(url)
    return responses[url]

doc = WebDocument("https://example.test/a", fake_fetcher)

assert doc.load_count == 0
assert doc.size == 5
assert doc.text == "alpha"
assert doc.load_count == 1
assert fetch_calls == ["https://example.test/a"]

# Cached.
assert doc.content == b"alpha"
assert doc.load_count == 1

# New URL invalidates.
doc.url = "https://example.test/b"
assert doc.text == "beta-beta"
assert doc.load_count == 2

# Explicit refresh forces another fetch.
doc.refresh()
assert doc.load_count == 3

print("Problem 9 passed")

Problem 9 passed


### Best-practice discussion

The example above is useful for understanding lazy properties, but real network access is often better exposed as an explicit method such as `load()` or `refresh()`.

Why?

- attribute access usually suggests a cheap operation
- network I/O can be slow
- network I/O can fail
- network access has observable external behavior

A good compromise is to make the expensive operation explicit, then expose already-loaded metadata through properties.

# Problem 10 — Refactor Expensive Property Access into an Explicit Load API

Refactor the previous idea into `RemoteResource`.

Rules:

1. Construction does **not** fetch anything.
2. `.load()` performs the fetch.
3. `.content`, `.size`, and `.text` are read-only properties.
4. Accessing those properties before loading raises `RuntimeError`.
5. Changing `.url` clears loaded state.
6. `.is_loaded` is a read-only Boolean property.

In [11]:
class RemoteResource:
    def __init__(self, url, fetcher, *, encoding="utf-8"):
        if not callable(fetcher):
            raise TypeError("fetcher must be callable")
        self._fetcher = fetcher
        self._encoding = encoding
        self._url = None
        self._content = None
        self.url = url

    @staticmethod
    def _validate_url(value):
        if not isinstance(value, str):
            raise TypeError("url must be a string")
        if not value.startswith(("http://", "https://")):
            raise ValueError("unsupported URL")

    @property
    def url(self):
        return self._url

    @url.setter
    def url(self, value):
        self._validate_url(value)
        if value != self._url:
            self._url = value
            self._content = None

    @property
    def is_loaded(self):
        return self._content is not None

    def load(self):
        data = self._fetcher(self.url)
        if not isinstance(data, bytes):
            raise TypeError("fetcher must return bytes")
        self._content = data
        return self

    def _require_loaded(self):
        if not self.is_loaded:
            raise RuntimeError("resource is not loaded; call load() first")

    @property
    def content(self):
        self._require_loaded()
        return self._content

    @property
    def size(self):
        return len(self.content)

    @property
    def text(self):
        return self.content.decode(self._encoding)


def fetcher(url):
    return f"payload from {url}".encode()

resource = RemoteResource("https://example.test/data", fetcher)

assert not resource.is_loaded

try:
    _ = resource.size
except RuntimeError:
    pass
else:
    raise AssertionError("size before load() should fail")

resource.load()

assert resource.is_loaded
assert resource.size > 0
assert "payload" in resource.text

resource.url = "https://example.test/other"
assert not resource.is_loaded

print("Problem 10 passed")

Problem 10 passed


# Problem 11 — Abstract Read-Only Property Contract

Use `abc.ABC` and `@abstractmethod` to define a `Shape` interface.

Requirements:

1. Every concrete shape must expose a read-only `area` property.
2. Create `RectangleShape` and `CircleShape`.
3. Add a normal concrete property `is_degenerate` to the base class.
4. Attempting to instantiate an incomplete subclass should fail.

In [12]:
from abc import ABC, abstractmethod
from math import pi

class Shape(ABC):
    @property
    @abstractmethod
    def area(self):
        """Return the shape's area."""
        raise NotImplementedError

    @property
    def is_degenerate(self):
        return self.area == 0


class RectangleShape(Shape):
    def __init__(self, width, height):
        if width < 0 or height < 0:
            raise ValueError("dimensions cannot be negative")
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height


class CircleShape(Shape):
    def __init__(self, radius):
        if radius < 0:
            raise ValueError("radius cannot be negative")
        self.radius = radius

    @property
    def area(self):
        return pi * self.radius ** 2


class BrokenShape(Shape):
    pass


assert RectangleShape(3, 4).area == 12
assert RectangleShape(0, 4).is_degenerate
assert CircleShape(2).area == 4 * pi

try:
    BrokenShape()
except TypeError:
    pass
else:
    raise AssertionError("abstract class contract was not enforced")

print("Problem 11 passed")

Problem 11 passed


# Problem 12 — Property Introspection and Documentation

Properties are descriptor objects stored on the class.

Create a `BankAccount` with:

- read-only `balance`
- read-only `is_overdrawn`
- a documented `owner` getter/setter

Then inspect the property objects directly.

This problem is about understanding the mechanics behind `@property`, not just using the syntax.

In [13]:
class BankAccount:
    def __init__(self, owner, opening_balance=0):
        self.owner = owner
        self._balance = float(opening_balance)

    @property
    def owner(self):
        """Name of the account owner."""
        return self._owner

    @owner.setter
    def owner(self, value):
        if not isinstance(value, str):
            raise TypeError("owner must be a string")
        value = value.strip()
        if not value:
            raise ValueError("owner cannot be empty")
        self._owner = value

    @property
    def balance(self):
        """Current account balance (read-only)."""
        return self._balance

    @property
    def is_overdrawn(self):
        """Whether the account balance is below zero."""
        return self.balance < 0

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self._balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("withdrawal must be positive")
        self._balance -= amount


account = BankAccount("  Grace Hopper  ", 100)

assert account.owner == "Grace Hopper"
assert account.balance == 100
assert not account.is_overdrawn

balance_descriptor = BankAccount.__dict__["balance"]
owner_descriptor = BankAccount.__dict__["owner"]

assert isinstance(balance_descriptor, property)
assert balance_descriptor.fget is not None
assert balance_descriptor.fset is None

assert owner_descriptor.fget is not None
assert owner_descriptor.fset is not None

assert "read-only" in BankAccount.balance.__doc__

print("Problem 12 passed")

Problem 12 passed


# Problem 13 — Thread-Safe Lazy Cache

A lazy cache can compute the same expensive value more than once when multiple threads access it simultaneously.

Implement `ThreadSafeReport.summary` using double-checked locking:

1. Fast path: return cached data without locking.
2. Slow path: acquire a lock.
3. Check the cache again after acquiring the lock.
4. Compute exactly once.
5. `replace_records()` changes the records and invalidates the cache safely.

This is an advanced concurrency example.

In [14]:
import threading
import time

_NOT_READY = object()

class ThreadSafeReport:
    def __init__(self, records):
        self._lock = threading.Lock()
        self._records = tuple(records)
        self._summary_cache = _NOT_READY
        self._computation_count = 0

    @property
    def records(self):
        return self._records

    def replace_records(self, records):
        new_records = tuple(records)
        with self._lock:
            self._records = new_records
            self._summary_cache = _NOT_READY

    @property
    def computation_count(self):
        return self._computation_count

    def _compute_summary(self):
        # Simulate expensive work.
        time.sleep(0.01)
        self._computation_count += 1
        values = self.records

        if not values:
            return {
                "count": 0,
                "min": None,
                "max": None,
                "mean": None,
            }

        return {
            "count": len(values),
            "min": min(values),
            "max": max(values),
            "mean": sum(values) / len(values),
        }

    @property
    def summary(self):
        # Fast path.
        if self._summary_cache is not _NOT_READY:
            return self._summary_cache

        # Slow path.
        with self._lock:
            # Another thread may have populated the cache while
            # this thread was waiting for the lock.
            if self._summary_cache is _NOT_READY:
                self._summary_cache = self._compute_summary()

            return self._summary_cache


report = ThreadSafeReport([1, 2, 3, 4, 5])
results = []

def worker():
    results.append(report.summary)

threads = [threading.Thread(target=worker) for _ in range(10)]

for thread in threads:
    thread.start()

for thread in threads:
    thread.join()

assert len(results) == 10
assert report.computation_count == 1
assert all(result["mean"] == 3 for result in results)

report.replace_records([10, 20])
assert report.summary["mean"] == 15
assert report.computation_count == 2

print("Problem 13 passed")

Problem 13 passed


# Problem 14 — Mini Project: Cached Statistical Series

Build a reusable `StatSeries` class.

Requirements:

- store immutable numeric values internally
- writable `values` property replaces the whole data set
- read-only `count`
- read-only `minimum`
- read-only `maximum`
- read-only `mean`
- read-only `variance` using population variance
- read-only `standard_deviation`
- read-only `range`
- expensive statistics should be cached
- replacing values invalidates all relevant caches
- empty series should have:
  - `count == 0`
  - all other statistics equal to `None`
- expose `cache_info` as a read-only diagnostic mapping
- callers must not be able to mutate internal cache state through `cache_info`

Try solving this before revealing the reference implementation below.

In [15]:
from math import sqrt
from numbers import Real
from types import MappingProxyType

_MISSING = object()

class StatSeries:
    STAT_NAMES = (
        "minimum",
        "maximum",
        "mean",
        "variance",
        "standard_deviation",
        "range",
    )

    def __init__(self, values=()):
        self._cache = {}
        self._compute_counts = {name: 0 for name in self.STAT_NAMES}
        self._values = ()
        self.values = values

    @staticmethod
    def _validated_values(values):
        values = tuple(values)

        for value in values:
            if isinstance(value, bool) or not isinstance(value, Real):
                raise TypeError("all values must be real numbers")

        return values

    def _invalidate_cache(self):
        self._cache.clear()

    def _cached(self, name, compute):
        value = self._cache.get(name, _MISSING)

        if value is _MISSING:
            value = compute()
            self._cache[name] = value
            self._compute_counts[name] += 1

        return value

    @property
    def values(self):
        return self._values

    @values.setter
    def values(self, values):
        validated = self._validated_values(values)

        if validated != self._values:
            self._values = validated
            self._invalidate_cache()

    @property
    def count(self):
        return len(self.values)

    @property
    def minimum(self):
        return self._cached(
            "minimum",
            lambda: min(self.values) if self.values else None,
        )

    @property
    def maximum(self):
        return self._cached(
            "maximum",
            lambda: max(self.values) if self.values else None,
        )

    @property
    def mean(self):
        return self._cached(
            "mean",
            lambda: (
                sum(self.values) / self.count
                if self.values
                else None
            ),
        )

    @property
    def variance(self):
        def compute():
            if not self.values:
                return None

            mean = self.mean
            return sum(
                (value - mean) ** 2
                for value in self.values
            ) / self.count

        return self._cached("variance", compute)

    @property
    def standard_deviation(self):
        return self._cached(
            "standard_deviation",
            lambda: (
                sqrt(self.variance)
                if self.variance is not None
                else None
            ),
        )

    @property
    def range(self):
        return self._cached(
            "range",
            lambda: (
                self.maximum - self.minimum
                if self.values
                else None
            ),
        )

    @property
    def cache_info(self):
        # Return snapshots/wrappers rather than mutable internal dicts.
        return MappingProxyType({
            "cached_keys": tuple(sorted(self._cache)),
            "compute_counts": MappingProxyType(dict(self._compute_counts)),
        })


series = StatSeries([1, 2, 3, 4, 5])

assert series.count == 5
assert series.minimum == 1
assert series.maximum == 5
assert series.mean == 3
assert series.variance == 2
assert series.standard_deviation == sqrt(2)
assert series.range == 4

# Re-accessing should use cached values.
_ = series.mean
_ = series.mean
_ = series.variance

info = series.cache_info
assert info["compute_counts"]["mean"] == 1
assert info["compute_counts"]["variance"] == 1

# Changing data invalidates cached values.
series.values = [10, 20, 30]

assert series.mean == 20
assert series.range == 20

# Empty behavior.
empty = StatSeries()
assert empty.count == 0
assert empty.minimum is None
assert empty.maximum is None
assert empty.mean is None
assert empty.variance is None
assert empty.standard_deviation is None
assert empty.range is None

print("Problem 14 passed")

Problem 14 passed


# Problem 15 — Challenge: Selective Cache Invalidation

Suppose a `Product` has:

- `base_price`
- `tax_rate`
- `shipping_weight`

Computed properties:

- `tax_amount = base_price * tax_rate`
- `price_with_tax = base_price + tax_amount`
- `shipping_cost = f(shipping_weight)`
- `checkout_total = price_with_tax + shipping_cost`

A naive design invalidates every cached value whenever anything changes.

Your task is to implement **selective invalidation**:

- changing `base_price` invalidates `tax_amount`, `price_with_tax`, `checkout_total`
- changing `tax_rate` invalidates `tax_amount`, `price_with_tax`, `checkout_total`
- changing `shipping_weight` invalidates `shipping_cost`, `checkout_total`
- unrelated caches remain intact

The solution below uses a tiny dependency map.

In [16]:
from numbers import Real

_UNSET = object()

class Product:
    DEPENDENCIES = {
        "base_price": {
            "tax_amount",
            "price_with_tax",
            "checkout_total",
        },
        "tax_rate": {
            "tax_amount",
            "price_with_tax",
            "checkout_total",
        },
        "shipping_weight": {
            "shipping_cost",
            "checkout_total",
        },
    }

    def __init__(self, base_price, tax_rate, shipping_weight):
        self._cache = {}
        self._compute_counts = {
            "tax_amount": 0,
            "price_with_tax": 0,
            "shipping_cost": 0,
            "checkout_total": 0,
        }

        self._base_price = _UNSET
        self._tax_rate = _UNSET
        self._shipping_weight = _UNSET

        self.base_price = base_price
        self.tax_rate = tax_rate
        self.shipping_weight = shipping_weight

    @staticmethod
    def _real(name, value, *, minimum=0):
        if isinstance(value, bool) or not isinstance(value, Real):
            raise TypeError(f"{name} must be a real number")
        if value < minimum:
            raise ValueError(f"{name} must be >= {minimum}")

    def _invalidate_for(self, source_name):
        for key in self.DEPENDENCIES[source_name]:
            self._cache.pop(key, None)

    def _set_input(self, public_name, private_name, value, validator):
        validator(value)

        current = getattr(self, private_name)
        if current is _UNSET or value != current:
            setattr(self, private_name, value)
            self._invalidate_for(public_name)

    def _cached(self, name, compute):
        if name not in self._cache:
            self._cache[name] = compute()
            self._compute_counts[name] += 1
        return self._cache[name]

    @property
    def base_price(self):
        return self._base_price

    @base_price.setter
    def base_price(self, value):
        self._set_input(
            "base_price",
            "_base_price",
            value,
            lambda x: self._real("base_price", x),
        )

    @property
    def tax_rate(self):
        return self._tax_rate

    @tax_rate.setter
    def tax_rate(self, value):
        def validate(x):
            self._real("tax_rate", x)
            if x > 1:
                raise ValueError("tax_rate must be between 0 and 1")
        self._set_input("tax_rate", "_tax_rate", value, validate)

    @property
    def shipping_weight(self):
        return self._shipping_weight

    @shipping_weight.setter
    def shipping_weight(self, value):
        self._set_input(
            "shipping_weight",
            "_shipping_weight",
            value,
            lambda x: self._real("shipping_weight", x),
        )

    @property
    def tax_amount(self):
        return self._cached(
            "tax_amount",
            lambda: self.base_price * self.tax_rate,
        )

    @property
    def price_with_tax(self):
        return self._cached(
            "price_with_tax",
            lambda: self.base_price + self.tax_amount,
        )

    @property
    def shipping_cost(self):
        def compute():
            # Example shipping model.
            if self.shipping_weight == 0:
                return 0.0
            return 5.0 + 1.25 * self.shipping_weight

        return self._cached("shipping_cost", compute)

    @property
    def checkout_total(self):
        return self._cached(
            "checkout_total",
            lambda: self.price_with_tax + self.shipping_cost,
        )

    @property
    def compute_counts(self):
        return dict(self._compute_counts)


p = Product(base_price=100, tax_rate=0.20, shipping_weight=4)

assert p.tax_amount == 20
assert p.price_with_tax == 120
assert p.shipping_cost == 10
assert p.checkout_total == 130

initial = p.compute_counts

# Changing only shipping weight should preserve price/tax caches.
p.shipping_weight = 8

assert "tax_amount" in p._cache
assert "price_with_tax" in p._cache
assert "shipping_cost" not in p._cache
assert "checkout_total" not in p._cache

assert p.shipping_cost == 15
assert p.checkout_total == 135

after_shipping = p.compute_counts

assert after_shipping["tax_amount"] == initial["tax_amount"]
assert after_shipping["price_with_tax"] == initial["price_with_tax"]
assert after_shipping["shipping_cost"] == initial["shipping_cost"] + 1
assert after_shipping["checkout_total"] == initial["checkout_total"] + 1

print("Problem 15 passed")

Problem 15 passed


# Additional Drill Problems — Try Without Looking Up a Solution

These are intentionally left without full implementations so you can use the solved problems above as patterns.

## Drill A — `Person.full_name`

Create a class with writable `first_name` and `last_name`, plus a read-only computed `full_name`.

Add normalization so leading/trailing spaces are stripped and names cannot be empty.

## Drill B — Writable `full_name`

Extend the previous class so assigning:

```python
person.full_name = "Ada Lovelace"
```

updates `first_name` and `last_name`.

Decide how to handle:

- one-word names
- multiple spaces
- three or more name components

Document your policy.

## Drill C — Cached Polynomial Evaluation

Create a polynomial object with coefficients and a writable `x`.

A read-only property `value` evaluates the polynomial at `x`.

Cache `value` until either the coefficients or `x` changes.

## Drill D — Cached Matrix Properties

For a 2×2 matrix, create read-only properties for:

- determinant
- trace
- transpose
- invertibility

Cache the expensive values and decide which should be invalidated when an element changes.

## Drill E — Immutable Configuration View

Build a configuration object that stores a mutable internal dictionary but exposes a read-only mapping through a property using `MappingProxyType`.

Explain why returning the original dictionary would not truly be read-only.

## Drill F — Time-To-Live Cache

Create a property whose cached value expires after a given number of seconds.

Inject a clock function so the class can be tested without sleeping.

## Drill G — Error Caching Policy

Suppose an expensive computation raises an exception.

Should your cache remember the exception or retry next time?

Implement both policies and compare them.

## Drill H — Dependency Graph

Generalize Problem 15 so dependencies are represented as a directed graph and invalidation propagates transitively.

For example:

```text
a ──> c ──> e
b ──> c
b ──> d ──> e
```

Changing `b` should invalidate `c`, `d`, and `e`.

# Best-Practice Checklist

Use this checklist when deciding whether to implement something as a property.

### Good candidates for properties

- values that conceptually belong to the object
- cheap computed values
- validation around attribute assignment
- derived values that preserve a simple attribute-like interface
- read-only public views of internal state

### Be careful with

- network access
- database queries
- filesystem work
- long-running computations
- hidden mutations
- surprising exceptions
- caches that are difficult to invalidate correctly

### Cache safely

- distinguish “not cached” from a legitimate `None`
- invalidate every dependent cached value
- avoid leaking mutable cached objects
- do not invalidate if the underlying input did not actually change
- consider version-based caching when many values depend on common state
- use locks if multiple threads can populate or invalidate shared caches
- consider `cached_property` when its semantics fit your object

### API design rule of thumb

If a caller should *notice the cost or side effect*, prefer a method.

If access is cheap, deterministic, and naturally attribute-like, a property is often appropriate.

# Final Review Questions

1. Why can a getter-only property still fail to make data absolutely immutable in Python?
2. What is the difference between a read-only property and an immutable object?
3. Why is `None` sometimes a bad cache sentinel?
4. When does `cached_property` need explicit invalidation?
5. Why can returning a cached list violate encapsulation?
6. What is selective invalidation?
7. Why might a version counter simplify cache management?
8. Why can lazy network access inside a property be surprising?
9. What problem does double-checked locking solve?
10. When is a writable derived property a good design?
11. How does a property object relate to Python's descriptor protocol?
12. Why should validation normally happen before mutating instance state?

# Suggested Mastery Exercise

Combine the ideas from Problems 8, 9, 13, and 15 into one class that:

- has multiple mutable inputs
- has several dependent computed properties
- caches expensive results
- uses a dependency graph or versioning strategy
- is safe under concurrent reads
- makes external I/O explicit through a method
- exposes diagnostic cache statistics as read-only data

If you can implement that cleanly and test all invalidation paths, you have moved well beyond basic `@property` usage.